# Revision Selection Options — Analysis Companion

## tl;dr

- The 144 frozen revisions have low negative development/validation rank persistence across profit, drawdown, and profit/drawdown.
- Option A preserves the nine-revision validation top-five union; Option B retains the five revisions that appear in at least two validation top-five lists; Option C retains the six revisions that are top 20 for the same metric in both windows.
- The options are decision frames, not a composite ranking or automatic promotion rule.
- Study OOS is not read or referenced by this notebook and remains closed.

## Context & Methods

This report companion recomputes all three independent ranks from the immutable development and validation `metrics.jsonl` artifacts using the locked SHA-256 tie-break. It verifies the metrics and ranking file hashes against each artifact manifest, joins all 144 revisions by immutable identity, and computes filled-trade-signal Jaccard overlap within each option.

### Key Assumptions

- Development and validation are the only admitted evidence windows.
- Option A is the union of the three validation top-five lists.
- Option B requires membership in at least two of the three validation top-five lists.
- Option C requires top-20 membership for the same metric in both development and validation.
- Filled-trade overlap is a setup-similarity diagnostic, not portfolio return correlation.
- No assumed trading costs are deducted.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

working = Path.cwd().resolve()
repo_root = next(
    path for path in (working, *working.parents)
    if (path / "pyproject.toml").exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from reports.revision_selection_options.analysis import build_analysis

analysis = build_analysis()

## Data

### 1. Validate the joined evidence

In [2]:
data_check = pd.DataFrame(
    [
        {
            "check": "Frozen revisions",
            "value": len(analysis.strategies),
        },
        {
            "check": "Aligned development/validation identities",
            "value": set(analysis.metrics["development"]) == set(analysis.metrics["validation"]),
        },
        {
            "check": "Development metrics rows",
            "value": len(analysis.metrics["development"]),
        },
        {
            "check": "Validation metrics rows",
            "value": len(analysis.metrics["validation"]),
        },
    ]
)
display(data_check)

,check,value
0,Frozen revisions,144
1,Aligned development/validation identities,True
2,Development metrics rows,144
3,Validation metrics rows,144


## Results

### 2. Recompute rank persistence and top-20 continuity

In [3]:
metric_labels = {
    "profit": "Gross profit",
    "drawdown": "Lowest maximum drawdown",
    "profit_drawdown": "Profit/drawdown",
}
stability = pd.DataFrame(
    [
        {
            "metric": metric_labels[metric],
            "spearman_rank_correlation": round(float(analysis.spearman[metric]), 4),
            "shared_top_20": len(analysis.shared_top20[metric]),
        }
        for metric in ("profit", "drawdown", "profit_drawdown")
    ]
)
display(stability)

,metric,spearman_rank_correlation,shared_top_20
0,Gross profit,-0.1855,2
1,Lowest maximum drawdown,-0.1642,3
2,Profit/drawdown,-0.2260,1


### 3. Construct the three exact option baskets

In [4]:
option_labels = {
    "A": "Validation frontier",
    "B": "Validation multi-metric core",
    "C": "Same-metric cross-window continuity",
}
option_summary = pd.DataFrame(
    [
        {
            "option": option,
            "label": option_labels[option],
            "revisions": len(analysis.options[option]),
            "development_mean_filled_trade_overlap": round(
                float(analysis.overlap[option]["development"]["mean"]), 4
            ),
            "validation_mean_filled_trade_overlap": round(
                float(analysis.overlap[option]["validation"]["mean"]), 4
            ),
            "validation_max_filled_trade_overlap": round(
                float(analysis.overlap[option]["validation"]["maximum"]), 4
            ),
        }
        for option in ("A", "B", "C")
    ]
)
display(option_summary)

,option,label,revisions,development_mean_filled_trade_overlap,validation_mean_filled_trade_overlap,validation_max_filled_trade_overlap
0,A,Validation frontier,9,0.0896,0.0888,0.4000
1,B,Validation multi-metric core,5,0.0426,0.0440,0.1795
2,C,Same-metric cross-window continuity,6,0.0489,0.0546,0.1765


### 4. Review the exact revisions in each option

In [5]:
def option_table(option):
    rows = []
    for identity in analysis.options[option]:
        rows.append(
            {
                "revision": analysis.strategies[identity]["name"],
                "identity_prefix": identity[:12],
                "development_ranks_P_DD_R": " / ".join(
                    str(analysis.ranks["development"][metric][identity])
                    for metric in ("profit", "drawdown", "profit_drawdown")
                ),
                "validation_ranks_P_DD_R": " / ".join(
                    str(analysis.ranks["validation"][metric][identity])
                    for metric in ("profit", "drawdown", "profit_drawdown")
                ),
                "development_profit": float(
                    analysis.metrics["development"][identity]["gross_profit"]
                ),
                "validation_profit": float(
                    analysis.metrics["validation"][identity]["gross_profit"]
                ),
                "validation_drawdown": float(
                    analysis.metrics["validation"][identity]["maximum_drawdown"]
                ),
                "validation_trades": analysis.metrics["validation"][identity]["trade_count"],
            }
        )
    return pd.DataFrame(rows).sort_values(
        ["validation_ranks_P_DD_R", "identity_prefix"]
    )

for option in ("A", "B", "C"):
    print(f"Option {option}: {option_labels[option]}")
    display(option_table(option))

Option A: Validation frontier


,revision,identity_prefix,development_ranks_P_DD_R,validation_ranks_P_DD_R,development_profit,validation_profit,validation_drawdown,validation_trades
5,monthly-ema6-below__close-cross-sma10__atr14x1...,8c6c38ba6f6c,44 / 110 / 54,1 / 70 / 11,11059.541014,10418.129621,0.034365,26
0,weekly-ema13-below__return5-cross-zero__rollin...,cd89dd0d61df,117 / 89 / 118,11 / 2 / 3,-518.234378,7111.937300,0.016101,26
2,monthly-ema6-above__return5-cross-zero__rollin...,553ee87a4395,76 / 56 / 70,13 / 5 / 4,5766.903298,7055.155404,0.017681,21
6,monthly-ema6-below__close-cross-sma10__rolling...,17e5de083ecb,110 / 126 / 112,2 / 82 / 15,1223.945305,9740.488161,0.037769,30
7,monthly-ema6-below__return5-cross-zero__rollin...,8562be0e3187,137 / 113 / 142,21 / 3 / 6,-5655.559132,6527.917234,0.017097,23
1,monthly-ema6-below__close-cross-sma10__atr14x1...,ac431397b274,120 / 88 / 122,3 / 8 / 2,-654.966432,9014.919450,0.019963,37
8,weekly-ema13-below__close-cross-ema5__atr14x1_...,ad135d51040e,20 / 117 / 44,4 / 20 / 5,15103.676826,8670.937070,0.022161,67
4,monthly-ema6-below__close-cross-sma10__rolling...,f466b2eb8f38,136 / 127 / 137,5 / 1 / 1,-4960.650436,8012.868408,0.014765,20
3,weekly-ema13-below__close-cross-sma10__atr14x1...,fc0b7c9e5a8a,90 / 124 / 97,58 / 4 / 24,3731.714617,3906.281105,0.017330,41


Option B: Validation multi-metric core


,revision,identity_prefix,development_ranks_P_DD_R,validation_ranks_P_DD_R,development_profit,validation_profit,validation_drawdown,validation_trades
0,weekly-ema13-below__return5-cross-zero__rollin...,cd89dd0d61df,117 / 89 / 118,11 / 2 / 3,-518.234378,7111.937300,0.016101,26
2,monthly-ema6-above__return5-cross-zero__rollin...,553ee87a4395,76 / 56 / 70,13 / 5 / 4,5766.903298,7055.155404,0.017681,21
1,monthly-ema6-below__close-cross-sma10__atr14x1...,ac431397b274,120 / 88 / 122,3 / 8 / 2,-654.966432,9014.919450,0.019963,37
4,weekly-ema13-below__close-cross-ema5__atr14x1_...,ad135d51040e,20 / 117 / 44,4 / 20 / 5,15103.676826,8670.937070,0.022161,67
3,monthly-ema6-below__close-cross-sma10__rolling...,f466b2eb8f38,136 / 127 / 137,5 / 1 / 1,-4960.650436,8012.868408,0.014765,20


Option C: Same-metric cross-window continuity


,revision,identity_prefix,development_ranks_P_DD_R,validation_ranks_P_DD_R,development_profit,validation_profit,validation_drawdown,validation_trades
0,monthly-ema6-above__return5-cross-zero__rollin...,7e3385bf5eaf,18 / 58 / 28,10 / 59 / 20,15804.956518,7219.875957,0.030535,27
5,monthly-ema6-above__close-cross-ema5__atr14x1p...,d0f874c6edc5,16 / 22 / 14,37 / 16 / 19,17344.320487,5185.585940,0.021254,37
4,weekly-ema13-below__close-cross-ema5__atr14x1_...,ad135d51040e,20 / 117 / 44,4 / 20 / 5,15103.676826,8670.937070,0.022161,67
3,weekly-ema13-above__close-cross-sma10__atr14x1...,bbc265401e3e,34 / 11 / 20,53 / 18 / 32,12062.638724,4287.615024,0.021776,50
2,weekly-ema13-above__return5-cross-zero__atr14x...,7c6de4378c99,8 / 12 / 6,61 / 6 / 29,21364.526715,3656.273342,0.017810,49
1,weekly-ema13-above__close-cross-sma10__atr14x1...,97c792f6be28,49 / 1 / 21,85 / 19 / 73,10123.658796,1897.378927,0.022021,30


## Takeaways

- No option dominates all independent metrics and both evidence windows.
- Option A maximizes validation-frontier breadth, Option B reduces that set to repeated validation top-five membership, and Option C substitutes same-metric cross-window continuity for validation-frontier coverage.
- A fourth defensible decision is to freeze no v1 revision and leave the study paused.
- The report does not create an OOS selection document, open OOS, or begin forward paper testing.